<img src="../../../Individual-Contest/Pixel/figs/IOAI-Logo.png" alt="Logo IOAI" width="200" height="auto">

[IOAI 2025 (Pekin, Chine), concours individuel](https://ioai-official.org/china-2025)

[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SKonteye/IOAI-2025/blob/main/fr/Individual-Contest/Pixel/Pixel.ipynb)


# Pixel Efficiency

## 1. Description du probleme

Vous etes etudiant en biologie de la faune et participez a un projet de recherche revolutionnaire au Starr Park Research Center. Votre equipe a deploye des milliers de pieges photographiques dans des zones sauvages isolees afin de surveiller les populations d'especes menacees. Cependant, les connexions Internet par satellite dans ces lieux reculs ont une bande passante extremement limitee. Votre mission est d'ecrire du code qui identifie les pixels les plus essentiels de chaque photographie d'animal, afin que seules les informations visuelles cruciales soient transmises au siege.

<img src="../../../Individual-Contest/Pixel/figs/Pixel Fig 1.png" width="400">

## 2. Jeu de donnees

Le jeu de donnees comprend un ensemble d'entrainement et un ensemble de test. Les jeux de donnees sont charges avec `load_from_disk` et utilisent le format `datasets`. L'ensemble de test n'est pas visible pour les concurrents.

Le jeu de donnees contient les champs suivants :

- `image` : les images sont en couleurs RGB au format PIL, chaque image a une taille de (224, 224)
- `name` : le label d'espece animale
- `idx` : identifiant unique permettant de retrouver les enregistrements.

1. **Ensemble d'entrainement (`train_dataset` folder)** :
    - Sert a entrainer vos modeles et a realiser des experimentations ; il est directement accessible et telechargeable pendant le concours.
    - Il contient 700 images.

2. **Ensemble de test (`test_dataset` folder)** :
    - Meme format que l'ensemble d'entrainement, mais sans le champ `name`.
    - Il contient 698 images, separees en deux jeux de test selon un ratio 3:7 : 30% des donnees servent a calculer le score du classement A, et 70% a calculer le score du classement B.
    - L'ensemble de test sert a calculer les scores des classements A et B et n'est pas directement accessible pendant le concours. Les concurrents peuvent consulter leur resultat sur le classement A, mais pas sur le classement B. Le score final est calcule uniquement a partir du classement B. Les sous-ensembles utilises pour les classements A et B sont totalement disjoints.
    

## 3. Tache
Vous disposez d'un jeu de donnees de photographies d'animaux et d'un modele CLIP capable de realiser une classification d'especes animales en zero-shot. Pour economiser de la bande passante, vous devez conserver au plus **6.25%** des pixels de chaque image, tout en maintenant la precision de classification aussi elevee que possible.

Plus precisement, votre tache consiste a renvoyer **un masque rectangulaire** pour chaque image, contenant une unique zone rectangulaire indiquant la zone a conserver. Chaque masque est defini par deux tuples de coordonnees : un pour le coin superieur gauche et un pour le coin inferieur droit du rectangle. Voici une visualisation de l'image apres application d'un masque rectangulaire produit par le baseline :

<img src="../../../Individual-Contest/Pixel/figs/Pixel Fig 2.png" width="400">

**Convention de coordonnees :**
- Les coordonnees du coin superieur gauche sont **inclusives** (le pixel a cette position est inclus dans le masque)
- Les coordonnees du coin inferieur droit sont **exclusives** (le pixel a cette position n'est PAS inclus dans le masque)

Par exemple, si vous indiquez les coordonnees `((10, 20), (15, 25))`, le masque couvre les pixels des lignes 10 a 14 (inclus) et des colonnes 20 a 24 (inclus), soit une surface totale de 25 pixels.

A titre d'illustration, si une image fait 3x3 et que l'on souhaite ne conserver que le pixel en haut a droite avec les coordonnees `((0, 2), (1, 3))`, le masque binaire resultant est :

```
[[0, 0, 1],
 [0, 0, 0],
 [0, 0, 0]]
```

Resume des exigences pour vos masques :

- Renvoyer un masque rectangulaire defini par des tuples de coordonnees : `((top, left), (bottom, right))`
- Les coordonnees du coin superieur gauche sont inclusives, celles du coin inferieur droit sont exclusives
- Le masque rectangulaire doit couvrir au plus *6.25%* des pixels d'origine (reduction minimale de 93.75%)
- Toutes les images sont de taille (224, 224), donc les valeurs de coordonnees doivent etre dans l'intervalle [0, 224]


Les images sont masquees a l'aide du masque que vous avez cree : en dehors du rectangle conserve, tous les pixels sont remplaces par la valeur RGB(0, 0, 0) (noir). L'image masquee est ensuite transmise au modele CLIP pour evaluation, et votre objectif est de maintenir la precision de classification de CLIP sur ces images masquees aussi elevee que possible. **Une classe supplementaire `other` est ajoutee aux classes a predire** afin de garantir que votre image masquee contient encore des informations reellement utiles pour les chercheurs du Starr Park Headquarters. Ainsi, si votre image masquee ne contient plus d'information animale, le modele predit la classe `other` au lieu de predire un animal au hasard et d'avoir une chance de tomber juste.

Vous ne devez utiliser que le modele CLIP et le jeu de donnees fournis. Pour rappel, CLIP genere des representations pour le texte et les images, et il peut calculer un score de similarite entre les deux. Ainsi, si vous disposez de dix classes animales, CLIP peut examiner une image et determiner quel texte (classe) lui est le plus proche.

Pour vous assurer que votre solution peut gerer le flux d'images du centre de recherche, votre code doit s'executer en **MOINS DE 8 MINUTES pour les 698 images de l'ensemble de test**. Il est recommande de tester votre solution sur l'ensemble d'entrainement (700 images) pour estimer le temps d'execution (l'ensemble de test prend legerement plus de temps a cause du chargement).

## 4. Soumission

Les concurrents doivent soumettre un notebook nomme `submission.ipynb`. Le fichier doit produire un fichier `.jsonl` nomme `submission.jsonl`, contenant tous les masques generes pour le split du jeu de donnees. Chaque masque dans `submission.jsonl` doit etre stocke comme un tuple de deux tuples de coordonnees : `((top, left), (bottom, right))`, ou le coin superieur gauche est inclusif et le coin inferieur droit exclusif.

Les concurrents n'ont pas besoin de separer les ensembles de test entre classement A et classement B : la machine d'evaluation lit `submission.jsonl` et calcule automatiquement les scores des classements A et B a partir des predictions et des labels reels.

Les fichiers de soumission doivent respecter strictement le format et le nommage ci-dessus, faute de quoi le systeme ne pourra pas les lire correctement.

## 5. Score

La metrique d'evaluation est la **precision de classification**, definie comme la proportion d'echantillons correctement predits parmi le total des echantillons evalues.

Votre score est la precision de classification zero-shot de CLIP sur les images de test masquees. **Si un masque soumis pour une image est invalide (forme incorrecte, plus de 6.25% de pixels conserves, etc.), cette image est comptee comme incorrecte. Un script d'exemple est fourni pour calculer le score sur le split d'entrainement.**


## 6. Baseline et ensemble d'entrainement

- Vous trouverez ci-dessous la solution baseline.
- Le jeu de donnees se trouve dans le dossier `training_set`.
- Le meilleur score du comite scientifique pour cette tache est 0.83 sur le classement B ; ce score est utilise pour l'unification des scores.
- Le score baseline du comite scientifique pour cette tache est 0.19 sur le classement B ; ce score est utilise pour l'unification des scores.


In [ ]:
# Installation des paquets requis
!pip install numpy pillow tqdm torch transformers datasets accelerate matplotlib

In [ ]:

from datasets import load_dataset
print("Chargement du jeu de donnees...")
# Charge le jeu de donnees directement depuis HuggingFace sans parametre data_dir
# Maniere correcte de charger un sous-ensemble/chemin specifique depuis un jeu de donnees HuggingFace

DATASET_NAME = "IOAI-official/IOAI-2025-Pixel-train"
SPLIT = 'train'
load_dataset(DATASET_NAME, data_dir="data", split=SPLIT) 

print(f"Jeu de donnees charge avec succes ! Total : {len(dataset)} echantillons")

# Affiche le premier element pour verifier les champs disponibles
print("\nCles du premier element :")
print(dataset[0].keys())

# Affiche quelques statistiques de base sans convertir les champs
print(f"\nJeu de donnees charge avec succes !")
print(f"Total : {len(dataset)} echantillons")

print(f"\nStructure d'un element :")
sample_item = dataset[0]
print(f"  Cles : {list(sample_item.keys())}")
if 'image' in sample_item:
    print(f"  Type d'image : {type(sample_item['image'])}")
    print(f"  Taille d'image : {sample_item['image'].size}")
else:
    print("  Aucun champ 'image' trouve dans le jeu de donnees")
    
if 'idx' in sample_item:
    print(f"  Index : {sample_item['idx']}")
else:
    print("  Aucun champ 'idx' trouve dans le jeu de donnees")



In [ ]:

print("Chargement du jeu de donnees...")
# Charge le jeu de donnees directement depuis HuggingFace sans parametre data_dir
# Maniere correcte de charger un sous-ensemble/chemin specifique depuis un jeu de donnees HuggingFace
dataset = load_dataset(DATASET_NAME, data_files="Individual-Contest/Pixel/training_set/train/data-00000-of-00001.arrow", split=SPLIT)

print(f"Jeu de donnees charge avec succes ! Total : {len(dataset)} echantillons")

# Affiche le premier element pour verifier les champs disponibles
print("\nCles du premier element :")
print(dataset[0].keys())

# Affiche quelques statistiques de base sans convertir les champs
print(f"\nJeu de donnees charge avec succes !")
print(f"Total : {len(dataset)} echantillons")

print(f"\nStructure d'un element :")
sample_item = dataset[0]
print(f"  Cles : {list(sample_item.keys())}")
if 'image' in sample_item:
    print(f"  Type d'image : {type(sample_item['image'])}")
    print(f"  Taille d'image : {sample_item['image'].size}")
else:
    print("  Aucun champ 'image' trouve dans le jeu de donnees")
    
if 'idx' in sample_item:
    print(f"  Index : {sample_item['idx']}")
else:
    print("  Aucun champ 'idx' trouve dans le jeu de donnees")



In [ ]:
import random
import numpy as np
import torch

seed = 42

random.seed(seed)                  # aleatoire integre de Python
np.random.seed(seed)               # NumPy
torch.manual_seed(seed)            # PyTorch (CPU)
torch.cuda.manual_seed(seed)       # PyTorch (un seul GPU)
torch.cuda.manual_seed_all(seed)   # PyTorch (tous les GPU)

# Assure un comportement deterministe
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

### Dependances et variables de configuration


In [ ]:
import os
import matplotlib.pyplot as plt
import numpy as np
from collections import Counter
from PIL import Image
from tqdm import tqdm
import glob
import json
import math
import torch
import matplotlib.pyplot as plt
from datasets import load_dataset, load_from_disk
from transformers import CLIPProcessor, CLIPModel
from PIL import Image
from tqdm.auto import tqdm 


# Configuration du jeu de donnees
DATASET_PATH = "IOAI-official/IOAI-2025-Pixel-train"
SPLIT = "train"

# Configuration du modele
MODEL_PATH = "openai/clip-vit-large-patch14"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BACKGROUND_CLASS = "other"


# Configuration des images et du masquage
HEIGHT = 224
WIDTH = 224
RETAIN_RATIO = 0.0625  # Conserve 6.25% des pixels
MEAN_COLOR = (0, 0, 0)  # Valeurs RGB moyennes pour les zones masquees


### Chargement du jeu de donnees

Commencons par charger le jeu de donnees et examiner son contenu :



In [ ]:

print("Chargement du jeu de donnees...")
# Charge le jeu de donnees directement depuis HuggingFace sans parametre data_dir
dataset = load_dataset(DATASET_PATH, split=SPLIT)

print(f"Jeu de donnees charge avec succes ! Total : {len(dataset)} echantillons")

# Affiche le premier element pour verifier les champs disponibles
print("\nCles du premier element :")
print(dataset[0].keys())

# Affiche quelques statistiques de base sans convertir les champs
print(f"\nJeu de donnees charge avec succes !")
print(f"Total : {len(dataset)} echantillons")

print(f"\nStructure d'un element :")
sample_item = dataset[0]
print(f"  Cles : {list(sample_item.keys())}")
print(f"  Type d'image : {type(sample_item['image'])}")
print(f"  Taille d'image : {sample_item['image'].size}")
print(f"  Index : {sample_item['idx']}")



In [ ]:
# Visualise les 10 premiers echantillons
fig, axes = plt.subplots(2, 5, figsize=(15, 8))
axes = axes.flatten()

print("Visualisation des 10 premiers echantillons...")

for i in range(10):
    sample = dataset[i]
    image = sample['image']
    label = sample['name']
    
    axes[i].imshow(image)
    axes[i].set_title(f"{label}\n", fontsize=12)
    axes[i].axis('off')

plt.tight_layout()
plt.suptitle('10 premiers echantillons du jeu de donnees', fontsize=16, y=1.02)
plt.show()


### Modele

Chargeons maintenant le modele et examinons quelques predictions :


In [ ]:
print(f"Chargement du modele CLIP et du processeur : {MODEL_PATH}...")
model = CLIPModel.from_pretrained(MODEL_PATH).to(DEVICE)
processor = CLIPProcessor.from_pretrained(MODEL_PATH)
print("Modele et processeur charges avec succes.")

In [ ]:
image = dataset[0]['image']
# Affiche l'image avec son vrai label
plt.figure(figsize=(8, 6))
plt.imshow(image)
plt.title(f"Image d'exemple\nVrai label : {dataset[0]['name']}", fontsize=14)
plt.axis('off')
plt.show()


labels = sorted(list(set(dataset['name']))) + [BACKGROUND_CLASS]
text_inputs = processor(text=labels, return_tensors="pt", padding=True).to(DEVICE)
image_processed = processor(images=image, return_tensors="pt").to(DEVICE)
pixel_values = image_processed['pixel_values']
outputs_full = model(pixel_values=pixel_values, **text_inputs)
logits_full = outputs_full.logits_per_image  # Forme : (1, num_styles)
predicted_index_full = logits_full.argmax(dim=-1).item()

print(f"Label predit : {labels[predicted_index_full]}")


### Baseline : une methode de masquage triviale

Nous allons implementer une solution de masquage triviale qui masque aleatoirement 90% des pixels.


In [ ]:
def generate_center_crop_coordinates(image):
    """
    Generate coordinates for a center crop mask.
    
    Returns:
        tuple: ((top, left), (bottom, right)) coordinates for the crop
    """
    H, W = image.size
    total_px = H * W
    k = int(total_px * RETAIN_RATIO)
    
    # Calcule la longueur du cote du carre
    side_length = int(np.sqrt(k))
    
    # Calcule les coordonnees du centre
    center_h, center_w = H // 2, W // 2
    
    # Calcule les bornes du decoupe
    half_side = side_length // 2
    top = max(0, center_h - half_side)
    left = max(0, center_w - half_side)
    bottom = min(H, top + side_length)
    right = min(W, left + side_length)
    
    return ((top, left), (bottom, right))

def generate_mask_from_coordinates(image, coordinates):
    """
    Generate a binary mask from crop coordinates.
    
    Parameters:
        image: PIL Image
        coordinates: tuple of ((top, left), (bottom, right))
    
    Returns:
        numpy array: Binary mask with 1s in the crop area
    """
    H, W = image.size
    mask = np.zeros((H, W), dtype=np.int8)
    
    (top, left), (bottom, right) = coordinates
    mask[top:bottom, left:right] = 1
    
    return mask

In [ ]:
def apply_mask_with_mean(image, mask, mean_rgb=MEAN_COLOR):
    """
    Apply arbitrary binary mask to image, replacing masked areas with mean values

    Parameters:
    - image: PIL Image (224x224)
    - mask: Binary numpy array or PIL Image (224x224) where 0 is the area to drop and 1 is the area to keep
    - mean_rgb: RGB mean values to use (default: from config)

    Returns: Modified PIL Image
    """
    # Convertit les images en tableaux numpy
    img_array = np.array(image).copy()

    # Garantit que le masque est un tableau numpy
    if isinstance(mask, Image.Image):
        mask_array = np.array(mask.convert('L')) > 127  # Conversion en binaire
    else:
        mask_array = mask > 0

    # Reshape le masque pour le broadcast avec RGB
    mask_3d = np.stack([mask_array] * 3, axis=2)

    # Convertit les valeurs moyennes dans [0, 255]
    mean_values = np.array([int(m * 255) for m in mean_rgb])
    # Applique le masque : remplace les zones a 0 (supprimees) par les valeurs moyennes, conserve les zones a 1
    img_array = np.where(mask_3d, img_array, mean_values.reshape(1, 1, 3))

    return Image.fromarray(img_array.astype(np.uint8))

In [ ]:
image = dataset[0]['image']
# Affiche l'image avec son vrai label
plt.figure(figsize=(8, 6))
plt.imshow(image)
plt.title(f"Image d'exemple\nVrai label : {dataset[0]['name']}", fontsize=14)
plt.axis('off')
plt.show()


labels = sorted(list(set(dataset['name']))) + [BACKGROUND_CLASS]
text_inputs = processor(text=labels, return_tensors="pt", padding=True).to(DEVICE)

mask = generate_mask_from_coordinates(image, generate_center_crop_coordinates(image))
image_masked = apply_mask_with_mean(image, mask)

plt.figure(figsize=(8, 6))
plt.imshow(image_masked)
plt.title(f"Image masquee\nVrai label : {dataset[0]['name']}", fontsize=14)
plt.axis('off')
plt.show()

image_processed = processor(images=image_masked, return_tensors="pt").to(DEVICE)
pixel_values = image_processed['pixel_values']
outputs_full = model(pixel_values=pixel_values, **text_inputs)
logits_full = outputs_full.logits_per_image  # Forme : (1, num_styles)
predicted_index_full = logits_full.argmax(dim=-1).item()

print(f"Label predit : {labels[predicted_index_full]}")

### Export des masques



In [ ]:
dataset = load_dataset("IOAI-official/IOAI-2025-Pixel-test", split="test")

In [ ]:
## Export des resultats et validation sur le jeu de donnees complet
RETAIN_RATIO = 0.0625

masks = {}
for item in tqdm(dataset):
    image = item['image']

    ## remplacez la generation du masque par votre fonction
    coordinates = generate_center_crop_coordinates(image)
    
    # inutile de modifier ce qui suit, c'est juste l'ecriture du fichier
    idx = item['idx']
    # Pour la validation, on a toujours besoin de generer le masque complet
    mask = generate_mask_from_coordinates(image, coordinates)
    assert mask.shape == (224, 224), "Le masque doit etre de taille 224x224"
    assert mask.sum() <= RETAIN_RATIO * 224 * 224, "Vous ne devez conserver que 6.25% des pixels"
    
    # Sauvegarde uniquement les coordonnees (topleft, bottomright) au lieu du masque complet
    masks[idx] = coordinates

# Sauvegarde au format JSONL (un objet JSON par ligne) - plus sur que pickle
with open('submission.jsonl', 'w') as f:
    for idx, coordinates in masks.items():
        json.dump({"idx": idx, "coordinates": coordinates}, f)
        f.write('\n')

print("Masques sauvegardes dans submission.jsonl")

### Validation


In [ ]:
# # Code de validation des masques generes

# def check_validity(coordinates):
#     """
#     Check if coordinates are valid according to the requirements.
#     Returns True if valid, False otherwise.
#     """
#     try:
#         # Check if coordinates is a tuple of two tuples
#         if not isinstance(coordinates, tuple) or len(coordinates) != 2:
#             print(f"Coordinates is not a tuple of two tuples")
#             return False
        
#         (top, left), (bottom, right) = coordinates
        
#         # Check if all coordinates are integers
#         if not all(isinstance(coord, (int, np.integer)) for coord in [top, left, bottom, right]):
#             print(f"Coordinates are not integers")
#             return False
        
#         # Check if coordinates are within image bounds
#         # For slicing mask[top:bottom, left:right], valid ranges are:
#         # top, left: [0, 223] (inclusive)
#         # bottom, right: [1, 224] (inclusive) since we need top < bottom and left < right
#         if not (0 <= top < 224 and 0 <= left < 224 and 1 <= bottom <= 224 and 1 <= right <= 224):
#             print(f"Coordinates are not within image bounds")
#             return False
        
#         # Check if top-left is actually top-left of bottom-right (proper ordering)
#         if not (top < bottom and left < right):
#             print(f"Top-left is not actually top-left of bottom-right")
#             return False
        
#         # Check that the crop area doesn't exceed RETAIN_RATIO
#         crop_area = (bottom - top) * (right - left)
#         max_area = RETAIN_RATIO * 224 * 224
#         if crop_area > max_area:
#             print(f"Crop area {crop_area} exceeds max area {max_area}")
#             return False
        
#         return True
#     except Exception:
#         return False




# def validate_masks(masks):
#     """Simple validation of generated masks on the dataset"""
#     correct = 0
#     total = 0

#     labels = sorted(list(set(dataset['name']))) + ['other']
#     text_inputs = processor(text=labels, return_tensors="pt", padding=True).to(DEVICE)

#     with torch.no_grad():
#         for item in tqdm(dataset, desc="Validating masks"):
#             idx = item['idx']
#             if idx not in masks:
#                 continue

#             if not check_validity(masks[idx]):
#                 continue
                
#             mask_coordinates = masks[idx]
#             image = item['image']
#             true_label = item['name']
            
#             # Apply mask to image
#             if image.mode != "RGB":
#                 image = image.convert("RGB")
            
#             mask = generate_mask_from_coordinates(image, mask_coordinates)

#             # Apply mask with mean color replacement
#             img_array = np.array(image).copy()
#             mask_array = mask > 0
#             mask_3d = np.stack([mask_array] * 3, axis=2)
#             mean_values = np.array([0, 0, 0])  # Black mean color
#             img_array = np.where(mask_3d, img_array, mean_values.reshape(1, 1, 3))
#             masked_image = Image.fromarray(img_array.astype(np.uint8))
            
#             # Get prediction on masked image
#             image_processed = processor(images=masked_image, return_tensors="pt").to(DEVICE)
#             pixel_values = image_processed['pixel_values']
#             outputs = model(pixel_values=pixel_values, **text_inputs)
#             logits = outputs.logits_per_image
#             predicted_idx = logits.argmax(dim=-1).item()
#             predicted_label = labels[predicted_idx]
            
#             # Check if prediction is correct
#             if predicted_label == true_label:
#                 correct += 1
#             total += 1
    
#     accuracy = correct / total if total > 0 else 0
#     print(f"Validation Results:")
#     print(f"Total samples: {total}")
#     print(f"Correct predictions: {correct}")
#     print(f"Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")
    
#     return accuracy

# # Run validation
# accuracy = validate_masks(masks)
